In [1]:
# Setup (standalone run): load splits, rebuild preprocessor + define pipelines
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_train = pd.read_csv("data/X_train.csv")
y_train = pd.read_csv("data/y_train.csv").squeeze("columns")
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)
pipe_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
pipe_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)),
])
print("Setup done:", X_train.shape)

Setup done: (5634, 18)


## Task 5: Model Training

**Method:** fit both pipelines from Task 4 on the training set only (`X_train`, `y_train`) — encoded features in, `Churn (1/0)` out.

**Why this way:**
- Each `Pipeline` refits its own copy of the preprocessor on train data, so no information leaks from test.
- Training both candidates (not just one) lets Task 6 compare fairly and pick the winner.
- Prerequisite: run Task 4's cell first so `pipe_lr`, `pipe_rf`, `X_train`, `y_train` exist.

In [2]:
# Task 5: train both candidates on the training set
pipe_lr.fit(X_train, y_train)
print("LogisticRegression trained.")
pipe_rf.fit(X_train, y_train)
print("RandomForest trained.")
print("Train accuracy — LR:", round(pipe_lr.score(X_train, y_train), 4),
      "| RF:", round(pipe_rf.score(X_train, y_train), 4))
print("(Train accuracy is indicative only — real comparison happens on test in Task 6.)")

LogisticRegression trained.
RandomForest trained.
Train accuracy — LR: 0.7528 | RF: 0.9975
(Train accuracy is indicative only — real comparison happens on test in Task 6.)
